In [4]:
# ============================================================
# STAGE 1: Document Ingestion + Chunking
# ============================================================
!pip install pypdf sentence-transformers faiss-cpu openai -q

from pypdf import PdfReader

PDF_PATH = "GESTURE X .pdf"
# --- 1. Load PDF and extract raw text ---
def extract_text(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text

raw_text = extract_text(PDF_PATH)
print(f"Extracted {len(raw_text)} characters from PDF")
print(raw_text[:500])  # sanity check

# --- 2. Chunk the text (fixed-size with overlap) ---
def chunk_text(text, chunk_size=800, overlap=150):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(raw_text)
print(f"\nTotal chunks created: {len(chunks)}")
print(f"\nSample chunk:\n{chunks[0][:300]}")

Extracted 43917 characters from PDF
GESTURE X
A PROJECT REPORT
Submitted by
in PROGRAM OF
STUDY
in partial fulfillment for the award of the degree of
BACHELOR OF TECHNOLOGY 
SCHOOL OF COMPUTING SCIENCE AND ENGINEERING
VIT BHOPAL UNIVERSITY KOTHRIKALAN,
SEHORE MADHYA PRADESH - 466114
DIVYANSHU SHARMA—23BCE10149
SHAAN MATHUR—23BCE10377
SWAYUM BANSAL—23BCE10466 OM
NILESH SAWKARE—23BCE11372
NAHAR SINGH RANA—23BCE10970
DECEMBER 2024
VIT BHOPAL UNIVERSITY, KOTHRIKALAN, SEHORE
MADHYA PRADESH – 466114
 
 PROGRAM CHAIR
Dr. Vikas Panthi
Sch

Total chunks created: 68

Sample chunk:
GESTURE X
A PROJECT REPORT
Submitted by
in PROGRAM OF
STUDY
in partial fulfillment for the award of the degree of
BACHELOR OF TECHNOLOGY 
SCHOOL OF COMPUTING SCIENCE AND ENGINEERING
VIT BHOPAL UNIVERSITY KOTHRIKALAN,
SEHORE MADHYA PRADESH - 466114
DIVYANSHU SHARMA—23BCE10149
SHAAN MATHUR—23BCE10377



In [5]:
# ============================================================
# STAGE 2: Embedding Creation + Vector Store (FAISS)
# ============================================================
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# --- 1. Load embedding model ---
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# --- 2. Convert all chunks into embeddings ---
chunk_embeddings = embed_model.encode(chunks, show_progress_bar=True)
chunk_embeddings = np.array(chunk_embeddings).astype('float32')

print(f"Embeddings shape: {chunk_embeddings.shape}")  # (num_chunks, embedding_dim)

# --- 3. Build FAISS index (L2 similarity) ---
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings)

print(f"FAISS index built with {index.ntotal} vectors")

# --- 4. Quick retrieval test ---
def retrieve(query, k=3):
    query_vec = embed_model.encode([query]).astype('float32')
    distances, indices = index.search(query_vec, k)
    results = [chunks[i] for i in indices[0]]
    return results

# Test it
test_results = retrieve("What is the accuracy of the model?")
print("\n--- Test retrieval ---")
for i, r in enumerate(test_results):
    print(f"\nChunk {i+1}:\n{r[:200]}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Embeddings shape: (68, 384)
FAISS index built with 68 vectors

--- Test retrieval ---

Chunk 1:
utput:Convert recognized signs into clear audible speech.
1.Performance:Achieve a processing rate of at least 30 FPS for real-time interaction.
2.Accuracy:Ensure at least 95% accuracy in normal condit

Chunk 2:
S
CHAPTER-4:
DESIGN METHODOLOGY AND ITS
NOVELTY
CHAPTER-2: 
RELATED WORK INVESTIGATION 
CHAPTER-1:
PROJECT DESCRIPTION AND OUTLINE 
TABLE OF CONTENTS 
5
6
7
8
9
CHAPTER
NO.
TITLE PAGE NO.
APPENDIX
REF

Chunk 3:
em Architecture 
The system follows a modular design:
Input Interface: Uses a camera to capture video frames.
Image Preprocessing: Handles hand segmentation, background removal, and resizing.
Gesture 


In [11]:
# ============================================================
# STAGE 3 (Local model version): Answer Generation using FLAN-T5
# No API key, no billing, no rate limits - runs locally in Colab
# ============================================================
!pip install -q sentencepiece
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# --- 1. Load a free local text-generation model ---
# flan-t5-base is small (~250MB), good enough for grounded QA
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model_gen = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

# --- 2. Build the RAG prompt using retrieved chunks ---
def build_prompt(question, retrieved_chunks):
    context = "\n\n".join(retrieved_chunks)
    prompt = f"""Answer the question using only the context below. If the answer isn't in the context, say "I don't have enough information."

Context:
{context}

Question: {question}
Answer:"""
    return prompt

# --- 3. Full RAG pipeline: retrieve + generate ---
def ask(question, k=3):
    retrieved_chunks = retrieve(question, k=k)  # from Stage 2
    prompt = build_prompt(question, retrieved_chunks)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    outputs = model_gen.generate(**inputs, max_new_tokens=150, do_sample=False)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# --- 4. Test it ---
question = "What is the main purpose of the GestureX project?"
answer = ask(question)
print(f"Q: {question}\nA: {answer}")

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Q: What is the main purpose of the GestureX project?
A: bridge the communication gap between the hearing and deaf communities


In [12]:
questions = [
    "What accuracy did the GestureX model achieve?",
    "What library is used for hand detection?",
    "What are the future recommendations for this project?",
    "Who are the team members of this project?"
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {ask(q)}\n")

Q: What accuracy did the GestureX model achieve?
A: 95% accuracy for static gestures and 85% for dynamic gestures

Q: What library is used for hand detection?
A: CNNs

Q: What are the future recommendations for this project?
A: Improve Real-time Speed. 7.1.2 Improve Real-time Speed. 7.1.3 Integrate with Wearables

Q: Who are the team members of this project?
A: Divyanshu Sharma (23BCE0149), Shaan Mathur (23BCE10377), Swayum Bansal (23BCE10466), Om Nilesh Sawkare (23BCE11372), Nahar Singh Rana (23BCE10970)

